# Chess Transformer Player — Step 1: Data Preparation

This notebook downloads and prepares training data from two sources:
- **Lichess Elite Database**: Games between 2500+ ELO players (no bullet games)
- **Local Stockfish**: Re-annotates positions with engine-optimal moves

**Output**: `final_training_data.csv` with columns `[fen, move, source]`

**Run on**: Google Colab (CPU is fine, no GPU needed for data prep)

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install python-chess -q
!apt-get install -q stockfish
!which stockfish

In [ ]:
# ── Verify Stockfish works ────────────────────────────────────
import chess
import chess.engine

engine = chess.engine.SimpleEngine.popen_uci("/usr/games/stockfish")
board  = chess.Board()
result = engine.play(board, chess.engine.Limit(depth=8))
print("Stockfish move:", result.move.uci())  # should print e2e4 or similar
engine.quit()

In [ ]:
# ── Download Lichess Elite Database (4 months) ────────────────
# Each file: ~70-90 MB compressed, 2500+ vs 2300+ ELO, no bullet
import urllib.request, zipfile, os
import chess.pgn
import pandas as pd
import logging

logging.getLogger("chess.pgn").setLevel(logging.CRITICAL)

MONTHS = [
    "https://database.nikonoel.fr/lichess_elite_2024-10.zip",
    "https://database.nikonoel.fr/lichess_elite_2024-06.zip",
    "https://database.nikonoel.fr/lichess_elite_2023-09.zip",
    "https://database.nikonoel.fr/lichess_elite_2022-11.zip",
]

os.makedirs("/content/elite_pgns", exist_ok=True)
pgn_files = []

for url in MONTHS:
    fname    = url.split("/")[-1]
    zip_path = f"/content/{fname}"
    month    = fname.replace("lichess_elite_", "").replace(".zip", "")

    if not os.path.exists(zip_path):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"  ✓ {os.path.getsize(zip_path)/1e6:.0f} MB")
    else:
        print(f"  Already downloaded: {fname}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        for name in z.namelist():
            if name.endswith('.pgn'):
                out_path = f"/content/elite_pgns/{month}.pgn"
                with z.open(name) as src, open(out_path, 'wb') as dst:
                    dst.write(src.read())
                pgn_files.append(out_path)
                print(f"  Extracted: {out_path}")

print(f"\nReady to parse {len(pgn_files)} PGN files")

In [ ]:
# ── Parse PGN files into (FEN, move) pairs ────────────────────
def parse_pgn_file(pgn_path, max_games=30_000):
    pairs, game_count, skipped = [], 0, 0
    with open(pgn_path, 'r', encoding='utf-8', errors='replace') as f:
        while game_count < max_games:
            try:
                game = chess.pgn.read_game(f)
            except Exception:
                skipped += 1
                continue
            if game is None:
                break
            board, game_pairs, valid = game.board(), [], True
            for move in game.mainline_moves():
                try:
                    if move not in board.legal_moves:
                        valid = False; break
                    game_pairs.append({'fen': board.fen(), 'move': move.uci()})
                    board.push(move)
                except Exception:
                    valid = False; break
            if valid and len(game_pairs) >= 10:
                pairs.extend(game_pairs)
                game_count += 1
    return pairs, game_count, skipped

all_pairs, total_games = [], 0
for pgn_path in pgn_files:
    print(f"Parsing {os.path.basename(pgn_path)}...")
    pairs, n_games, n_skip = parse_pgn_file(pgn_path, max_games=30_000)
    all_pairs.extend(pairs)
    total_games += n_games
    print(f"  ✓ {n_games:,} games | {len(pairs):,} positions | {n_skip:,} skipped")

print(f"\nTotal: {total_games:,} games | {len(all_pairs):,} positions")

In [ ]:
# ── Deduplicate and save elite dataset ────────────────────────
df = pd.DataFrame(all_pairs)
print(f"Before dedup: {len(df):,}")
df = df.drop_duplicates(subset=['fen'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"After dedup:  {len(df):,}")

# Validate 500 random samples
errors = 0
for _, row in df.sample(500).iterrows():
    try:
        board = chess.Board(row['fen'])
        move  = chess.Move.from_uci(row['move'])
        if move not in board.legal_moves:
            errors += 1
    except:
        errors += 1
print(f"Validation: {errors} errors {'✓' if errors == 0 else '⚠️'}")

df.to_csv('/content/lichess_elite_training_data.csv', index=False)
print(f"Saved: {len(df):,} unique (fen, move) pairs")

In [ ]:
# ── Annotate 200k positions with local Stockfish ──────────────
# Replaces GM moves with engine-optimal moves for higher quality labels
df = pd.read_csv('/content/lichess_elite_training_data.csv')
df_sample = df.sample(200_000, random_state=42).reset_index(drop=True)

engine = chess.engine.SimpleEngine.popen_uci("/usr/games/stockfish")
engine.configure({"Threads": 2})

annotated, errors = [], 0
CHECKPOINT = '/content/stockfish_annotated.csv'

for i, row in df_sample.iterrows():
    try:
        board  = chess.Board(row['fen'])
        result = engine.play(board, chess.engine.Limit(depth=8))
        annotated.append({
            'fen'   : row['fen'],
            'move'  : result.move.uci(),
            'source': 'stockfish_annotated'
        })
    except Exception:
        errors += 1

    if (i + 1) % 20_000 == 0:
        pd.DataFrame(annotated).to_csv(CHECKPOINT, index=False)
        print(f"  {i+1:,} / 200,000 | errors: {errors} | checkpoint saved ✓")

engine.quit()
df_annotated = pd.DataFrame(annotated)
df_annotated.to_csv(CHECKPOINT, index=False)
print(f"Done: {len(df_annotated):,} positions annotated, {errors} errors")

In [ ]:
# ── Combine: replace GM moves with Stockfish where available ──
df_elite     = pd.read_csv('/content/lichess_elite_training_data.csv')
df_stockfish = pd.read_csv('/content/stockfish_annotated.csv')

df_elite['source'] = 'gm_elite'
sf_lookup = df_stockfish.set_index('fen')['move']

mask = df_elite['fen'].isin(sf_lookup.index)
df_elite.loc[mask, 'move']   = df_elite.loc[mask, 'fen'].map(sf_lookup)
df_elite.loc[mask, 'source'] = 'stockfish_annotated'

print(f"Stockfish replaced GM moves in: {mask.sum():,} positions")
print(f"GM-only positions:              {(~mask).sum():,}")

df_final = df_elite.sample(frac=1, random_state=42).reset_index(drop=True)
df_final.to_csv('/content/final_training_data.csv', index=False)

print(f"\nFinal dataset: {len(df_final):,} rows")
print(df_final['source'].value_counts())
print(df_final.sample(3).to_string())

In [ ]:
# ── Save to Google Drive (survives runtime restarts) ──────────
from google.colab import drive
drive.mount('/content/drive')

import shutil
os.makedirs('/content/drive/MyDrive/chess_data', exist_ok=True)

for fname in ['final_training_data.csv', 'stockfish_annotated.csv',
              'lichess_elite_training_data.csv']:
    shutil.copy(f'/content/{fname}',
                f'/content/drive/MyDrive/chess_data/{fname}')
    print(f"Saved {fname} to Drive ✓")